## Bussiness understanding

Our business requirement is to find semantically similar documents based on user-provided sentences.

## Data Collection

### Import necessary libraries

In [55]:
import os
import re
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

In [61]:
documents = []

DATASET_PATH = Path(r"R:\Learning\Core_NLP\02_Word2Vec\Projects\Document_similarity-engine\News Articles")

# Read all TXT files from category folders

for file_path in DATASET_PATH.rglob("*.txt"):

    # Extract document ID from filename
    document_id = file_path.stem

    # Extract category from parent folder
    category = file_path.parent.name

    # Read article content
    with open(file_path, "r", encoding="utf-8", errors="replace") as file:
        article = file.read()

    # Store collected information
    documents.append({
        "document_id": document_id,
        "category": category,
        "article": article,
        "document_key": f"{category}_{document_id}"
    })

# Convert collected data into DataFrame
document_data = pd.DataFrame(documents)

print("\nData collection completed!")


Data collection completed!


In [62]:
document_data

,document_id,category,article,document_key
0,001,business,Ad sales boost Time Warner profit\n\nQuarterly...,business_001
1,002,business,Dollar gains on Greenspan speech\n\nThe dollar...,business_002
2,003,business,Yukos unit buyer faces loan claim\n\nThe owner...,business_003
3,004,business,High fuel prices hit BA's profits\n\nBritish A...,business_004
4,005,business,Pernod takeover talk lifts Domecq\n\nShares in...,business_005
...,...,...,...,...
95,016,tech,Global blogger action day called\n\nThe global...,tech_016
96,017,tech,Finding new homes for old phones\n\nRe-using o...,tech_017
97,018,tech,PlayStation 3 chip to be unveiled\n\nDetails o...,tech_018
98,019,tech,Intel unveils laser breakthrough\n\nIntel has ...,tech_019


In [63]:
print("\nFirst 5 records:")
print(document_data.head())

print("\nTotal documents collected:")
print(len(document_data))

print("\nCategory names:")
print(document_data["category"].unique())


First 5 records:
  document_id  category                                            article  \
0         001  business  Ad sales boost Time Warner profit\n\nQuarterly...   
1         002  business  Dollar gains on Greenspan speech\n\nThe dollar...   
2         003  business  Yukos unit buyer faces loan claim\n\nThe owner...   
3         004  business  High fuel prices hit BA's profits\n\nBritish A...   
4         005  business  Pernod takeover talk lifts Domecq\n\nShares in...   

   document_key  
0  business_001  
1  business_002  
2  business_003  
3  business_004  
4  business_005  

Total documents collected:
100

Category names:
<StringArray>
['business', 'entertainment', 'politics', 'sport', 'tech']
Length: 5, dtype: str


## Data Understanding

### Exploratory Data Analysis

In [64]:
# 1. Total number of documents
print("\nTotal Documents:")
print(len(document_data))


# 2. Category-wise document distribution
print("\nCategory Distribution:")
print(document_data["category"].value_counts())


# 3. Identify empty documents
empty_documents = document_data[
    document_data["article"].fillna("").str.strip() == ""]

print("\nEmpty Documents:")
print(len(empty_documents))


# 4. Identify duplicate documents
duplicate_documents = document_data[
    document_data["article"].fillna("").str.strip().duplicated(keep=False)
    & (df["article"].fillna("").str.strip() != "")
]

print("\nDuplicate Documents:")
print(len(duplicate_documents))


# Display duplicate records
print("\nDuplicate Document Details:")
print(duplicate_documents[["document_id", "category"]])



Total Documents:
100

Category Distribution:
category
business         20
entertainment    20
politics         20
sport            20
tech             20
Name: count, dtype: int64

Empty Documents:
0

Duplicate Documents:
2

Duplicate Document Details:
   document_id category
71         012    sport
79         020    sport


## Data Preparation

In [65]:
# Remove the Duplicate document

document_data = document_data[
    df["article"].fillna("").str.strip() != ""
].copy()

# Reset the index values
document_data.reset_index(drop=True, inplace=True)

### Data Cleaning

In [68]:
stop_words = set(stopwords.words("english"))
stop_words = stop_words - {"no", "not", "never"}

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Sentence tokenization
    sentences = sent_tokenize(text)

    processed_sentences = []

    for sentence in sentences:

        # Remove numbers and unwanted characters
        sentence = re.sub(r"[^a-z\s]", " ", sentence)

        # Remove extra whitespace
        sentence = re.sub(r"\s+", " ", sentence).strip()

        # Word tokenization
        words = word_tokenize(sentence)

        # Stopword removal
        words = [
            word for word in words
            if word not in stop_words and len(word) > 1
        ]

        # Keep sentences containing meaningful words
        if words:
            processed_sentences.append(words)

    return processed_sentences


document_data["processed_sentences"] = document_data["article"].apply(preprocess_text)

print("\nPreprocessing completed!")


Preprocessing completed!


In [69]:
sample = "Artificial Intelligence is NOT replacing human. Do you think it will change the humans ?"

result = preprocess_text(sample)

print(result)

[['artificial', 'intelligence', 'not', 'replacing', 'human'], ['think', 'change', 'humans']]


## Model Building & Training

In [70]:
training_corpus = [
    sentence
    for document_sentences in document_data["processed_sentences"]
    for sentence in document_sentences
]

model = Word2Vec(
    sentences=training_corpus,
    vector_size=50,
    window=3,
    min_count=2,
    sg=0,
    workers=4,
    seed=42
)


print("\nWord2Vec model created & Trained successfully!")


Word2Vec model created & Trained successfully!


## Model Training

## Document vector generation

In [71]:
def get_document_vector(processed_sentences, model):

    # Combine all words from the document's sentences
    words = [
        word
        for sentence in processed_sentences
        for word in sentence
    ]

    # Retrieve only words available in Word2Vec vocabulary
    word_vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    # Handle documents without known vocabulary words
    if not word_vectors:
        return None

    # Calculate average word embedding
    document_vector = np.mean(word_vectors, axis=0)

    return document_vector


document_data["document_vector"] = document_data["processed_sentences"].apply(
    lambda sentences: get_document_vector(sentences, model)
)

valid_documents = document_data[
    df["document_vector"].notna()
].copy()


# Reset index
valid_documents.reset_index(drop=True, inplace=True)


# ============================================
# CREATE DOCUMENT VECTOR DATAFRAME
# ============================================

document_vectors_df = valid_documents[
    ["document_key", "document_id", "category", "document_vector"]
].copy()

valid_documents
#document_vectors_df

,document_id,category,article,document_key,processed_sentences,document_vector
0,001,business,Ad sales boost Time Warner profit\n\nQuarterly...,business_001,"[[ad, sales, boost, time, warner, profit, quar...","[-0.000638, -0.007095, -0.000755, 0.000342, 0...."
1,002,business,Dollar gains on Greenspan speech\n\nThe dollar...,business_002,"[[dollar, gains, greenspan, speech, dollar, hi...","[-0.00079, -0.006646, 0.000485, 0.001995, -0.0..."
2,003,business,Yukos unit buyer faces loan claim\n\nThe owner...,business_003,"[[yukos, unit, buyer, faces, loan, claim, owne...","[0.000109, -0.008696, -0.001301, 0.002918, 0.0..."
3,004,business,High fuel prices hit BA's profits\n\nBritish A...,business_004,"[[high, fuel, prices, hit, ba, profits, britis...","[-0.00014, -0.007218, -0.001495, 0.001018, -0...."
4,005,business,Pernod takeover talk lifts Domecq\n\nShares in...,business_005,"[[pernod, takeover, talk, lifts, domecq, share...","[0.000131, -0.007788, -0.002533, 0.00256, -0.0..."
...,...,...,...,...,...,...
95,016,tech,Global blogger action day called\n\nThe global...,tech_016,"[[global, blogger, action, day, called, global...","[0.001576, -0.009247, -0.002231, 0.000348, 0.0..."
96,017,tech,Finding new homes for old phones\n\nRe-using o...,tech_017,"[[finding, new, homes, old, phones, using, old...","[-0.000747, -0.012359, -0.00372, 0.001534, -0...."
97,018,tech,PlayStation 3 chip to be unveiled\n\nDetails o...,tech_018,"[[playstation, chip, unveiled, details, chip, ...","[-0.001749, -0.008222, 0.000481, 0.003592, -0...."
98,019,tech,Intel unveils laser breakthrough\n\nIntel has ...,tech_019,"[[intel, unveils, laser, breakthrough, intel, ...","[-0.002049, -0.00747, -0.00182, 0.003907, 0.00..."


## Model Testing


In [72]:
# Convert document vectors into a NumPy matrix
document_matrix = np.vstack(
    document_vectors_df["document_vector"].values)

def search_documents(input_text, top_n=5):

     # Step 1: Preprocess input sentence
    processed_input = preprocess_text(input_text)

    # Step 2: Generate input document vector
    input_vector = get_document_vector(processed_input, model)

    if input_vector is None:
        print("\nNo known vocabulary words found in input.")
        return pd.DataFrame()
        
     # Step 3 : Calcullate cosine similarity
    scores = cosine_similarity(input_vector.reshape(1, -1),document_matrix)[0]

    # Step 4: Create results DataFrame
    results = document_vectors_df[
        ["document_id", "category", "document_key"]
    ].copy()
    results["similarity_score"] = scores


    # Step 5: Sort by similarity score
    results = results.sort_values(
        by="similarity_score",
        ascending=False
    )

    # Step 6: Return Top N documents
    results = results.head(top_n).reset_index(drop=True)
    return results


In [73]:
search_documents("intel silicon laser")

,document_id,category,document_key,similarity_score
0,019,tech,tech_019,0.488739
1,010,entertainment,entertainment_010,0.466833
2,006,business,business_006,0.459789
3,014,tech,tech_014,0.449297
4,018,tech,tech_018,0.448747
